In [1]:
import pandas as pd
import sys

# load data set
df = pd.read_csv("../NPFC-Test_Database_V2.csv")

relevant_columns = [
    "Subject_ID",
    "Timestamp",
    "Test_Time",
    "Task_Num",
    "Task_Time",
    "Task_Type",
    "Frame",
    "Task_Frame",
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Face_Detection",
    "resmasknet_anger",
    "resmasknet_disgust",
    "resmasknet_fear",
    "resmasknet_happiness",
    "resmasknet_sadness",
    "resmasknet_surprise",
    "resmasknet_neutral",
    "Temperature",
    "EDA",
    "BVP",
    "HeadBandOn",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
    "Gender",
    "Perceived_Tiredness",
    "Perceived_Stress",
    "Wearing_Glasses",
]

df = df[relevant_columns]

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47905 entries, 0 to 47904
Data columns (total 47 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Subject_ID            47905 non-null  object 
 1   Timestamp             47905 non-null  object 
 2   Test_Time             47905 non-null  object 
 3   Task_Num              47905 non-null  float64
 4   Task_Time             47905 non-null  object 
 5   Task_Type             47905 non-null  int64  
 6   Frame                 47905 non-null  int64  
 7   Task_Frame            47905 non-null  int64  
 8   Selfreport_valence    47905 non-null  int64  
 9   Selfreport_arousal    47905 non-null  int64  
 10  Selfreport_focus      47905 non-null  int64  
 11  Face_Detection        47905 non-null  int64  
 12  resmasknet_anger      46606 non-null  float64
 13  resmasknet_disgust    46606 non-null  float64
 14  resmasknet_fear       46606 non-null  float64
 15  resmasknet_happines

In [2]:
sys.path.append("./data-preprocessing")
from data_preparation import (
    prepare_data,
    normalize_signals_with_mediation_baseline,
    align_lag_signals,
)

# Clean, normalize and align data based on exploratory data analysis and domain knowledge
df = normalize_signals_with_mediation_baseline(df)
df = align_lag_signals(df)
df = prepare_data(df)

# Compare number of rows before and after!
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22177 entries, 14 to 47904
Data columns (total 49 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Subject_ID                    22177 non-null  object 
 1   Timestamp                     22177 non-null  object 
 2   Test_Time                     22177 non-null  object 
 3   Task_Num                      22177 non-null  float64
 4   Task_Time                     22177 non-null  object 
 5   Task_Type                     22177 non-null  int64  
 6   Frame                         22177 non-null  int64  
 7   Task_Frame                    22177 non-null  int64  
 8   Selfreport_valence            22177 non-null  int64  
 9   Selfreport_arousal            22177 non-null  int64  
 10  Selfreport_focus              22177 non-null  int64  
 11  Face_Detection                22177 non-null  int64  
 12  resmasknet_anger              22177 non-null  float64
 13  resma

# Feature selection

Selecting the best features (X).
Also creating the features for prediction (Y).


In [3]:
from feature_selection import variance_treshold_selection

# Grab resmasknet_dominant_emotion if you also want the emotion
result_df_regression = df["resmasknet_max_emotion_value"]

# Select relevant features
feature_columns = [
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Temperature",
    "EDA",
    "BVP",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
    "Gender",
    "Perceived_Tiredness",
    "Perceived_Stress",
    "Wearing_Glasses",
]

feature_df = df[feature_columns]
feature_df = variance_treshold_selection(feature_df)

# Output the results of the feature selection
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22177 entries, 14 to 47904
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Selfreport_valence   22177 non-null  int64  
 1   Selfreport_arousal   22177 non-null  int64  
 2   Selfreport_focus     22177 non-null  int64  
 3   Temperature          21902 non-null  float64
 4   EDA                  21902 non-null  float64
 5   BVP                  21902 non-null  float64
 6   Delta_TP9            22177 non-null  float64
 7   Delta_AF7            22177 non-null  float64
 8   Delta_AF8            22177 non-null  float64
 9   Delta_TP10           22177 non-null  float64
 10  Theta_TP9            22177 non-null  float64
 11  Theta_AF7            22177 non-null  float64
 12  Theta_AF8            22177 non-null  float64
 13  Theta_TP10           22177 non-null  float64
 14  Alpha_TP9            22177 non-null  float64
 15  Alpha_AF7            22177 non-null  flo

# Model Training

**CURRENTLY WHAT'S SHOWN IN THE OUTPUT IS BAD MODELS WITH VERY SMALL SIZE.**

- Ideally, uncomment the lines prefixed with _# BIG COMPUTE:_ to actually get the real model.
- This was done just to test the pipeline


In [11]:
# This is added because when testing with low iterations, some models throw convergence warnings. Annoying
# Remove when training real models
import warnings

warnings.filterwarnings("ignore")

In [4]:
### Data normalization preprocessor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler

# All physiological signals, brainwave bands, and temperature
sensor_and_eeg_features = [
    "Temperature",
    "EDA",
    "BVP",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
]

# Subjective human ratings
survey_features = [
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Perceived_Tiredness",
    "Perceived_Stress",
]

# Metadata/Demographics left untouched
binary_features = ["Gender", "Wearing_Glasses"]

# Robust pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("robust_sensors", RobustScaler(), sensor_and_eeg_features),
        ("minmax_survey", MinMaxScaler(), survey_features),
        ("pass_binary", "passthrough", binary_features),
    ]
)

# HOW TO USE:
# X_train_scaled = preprocessor.fit_transform(X_train)
# X_test_scaled = preprocessor.transform(X_test)

## Regression models

Here we are tranining the regression models. That is, models that will predict the uncertainty of the resmasknet FER model.


In [5]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Convert all variables to float to ensure missing values are numpy safe (from pandas.NA to np.nan)
# and also filling any created NaN to avoid downstream scikit-learn estimator failures.
X = feature_df.astype(float).fillna(0)
y = result_df_regression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Apply preprocessor perfectly matching the comment block example
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# Extract generated feature names dynamically
feature_names_out = preprocessor.get_feature_names_out()


In [ ]:
# --- Random Forest Regressor with RFE Backward Elimination ---
print("--- Random Forest Regressor ---")
# BIG COMPUTE: rf_reg = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf_reg = RandomForestRegressor(random_state=42, n_estimators=1, n_jobs=-1)
# RFECV performs recursive backward elimination and selected best features by CV score
rf_selector = RFECV(estimator=rf_reg, step=0.1, cv=3, scoring="neg_mean_squared_error")
rf_selector.fit(X_train_scaled, y_train)

X_train_rf = rf_selector.transform(X_train_scaled)
X_test_rf = rf_selector.transform(X_test_scaled)

rf_reg.fit(X_train_rf, y_train)
y_pred_rf = rf_reg.predict(X_test_rf)

print(f"Optimal number of features: {rf_selector.n_features_}")
print(f"Selected features: {feature_names_out[rf_selector.support_]}")

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
n_rf = X_test_rf.shape[0]  # type: ignore
p_rf = X_test_rf.shape[1]  # type: ignore
adj_r2_rf = (
    1 - (1 - r2_rf) * (n_rf - 1) / (n_rf - p_rf - 1)
    if (n_rf - p_rf - 1) > 0
    else float("nan")
)

print(f"MSE: {mse_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R-squared: {r2_rf:.4f}")
print(f"Adjusted R-squared: {adj_r2_rf:.4f}\n")

--- Random Forest Regressor ---
Optimal number of features: 3
Selected features: ['robust_sensors__Temperature' 'robust_sensors__EDA'
 'robust_sensors__Gamma_AF8']
MSE: 0.0308
RMSE: 0.1756
MAE: 0.1330
R-squared: -0.1482
Adjusted R-squared: -0.1490



In [12]:
# --- Neural Network Regressor with SFS Backward Elimination ---
print("--- Neural Network Regressor ---")
# BIG COMPUTE: nn_reg = MLPRegressor(random_state=42, max_iter=1000, hidden_layer_sizes=(64, 32))
nn_reg = MLPRegressor(random_state=42, max_iter=10, hidden_layer_sizes=(2, 2))
# SequentialFeatureSelector allows backward elimination on models without feature_importances_
try:
    # Drops features that do not improve performance by >0.001
    nn_selector = SequentialFeatureSelector(
        nn_reg,
        n_features_to_select="auto",
        tol=0.001,
        direction="backward",
        cv=3,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    nn_selector.fit(X_train_scaled, y_train)
except TypeError:
    # Fallback to keep 70% of features best for the NN
    nn_selector = SequentialFeatureSelector(
        nn_reg,
        n_features_to_select=max(1, int(X_train_scaled.shape[1] * 0.7)),
        direction="backward",
        cv=3,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    nn_selector.fit(X_train_scaled, y_train)

X_train_nn = nn_selector.transform(X_train_scaled)
X_test_nn = nn_selector.transform(X_test_scaled)

nn_reg.fit(X_train_nn, y_train)
y_pred_nn = nn_reg.predict(X_test_nn)

print(f"Selected features count: {nn_selector.get_support().sum()}")
print(f"Selected features: {feature_names_out[nn_selector.get_support()]}")

mse_nn = mean_squared_error(y_test, y_pred_nn)
rmse_nn = np.sqrt(mse_nn)
mae_nn = mean_absolute_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)
n_nn = X_test_nn.shape[0]
p_nn = X_test_nn.shape[1]
adj_r2_nn = (
    1 - (1 - r2_nn) * (n_nn - 1) / (n_nn - p_nn - 1)
    if (n_nn - p_nn - 1) > 0
    else float("nan")
)

print(f"MSE: {mse_nn:.4f}")
print(f"RMSE: {rmse_nn:.4f}")
print(f"MAE: {mae_nn:.4f}")
print(f"R-squared: {r2_nn:.4f}")
print(f"Adjusted R-squared: {adj_r2_nn:.4f}\n")

--- Neural Network Regressor ---
Selected features count: 29
Selected features: ['robust_sensors__Temperature' 'robust_sensors__EDA' 'robust_sensors__BVP'
 'robust_sensors__Delta_TP9' 'robust_sensors__Delta_AF7'
 'robust_sensors__Delta_AF8' 'robust_sensors__Delta_TP10'
 'robust_sensors__Theta_TP9' 'robust_sensors__Theta_AF7'
 'robust_sensors__Theta_AF8' 'robust_sensors__Theta_TP10'
 'robust_sensors__Alpha_TP9' 'robust_sensors__Alpha_AF7'
 'robust_sensors__Alpha_AF8' 'robust_sensors__Alpha_TP10'
 'robust_sensors__Beta_TP9' 'robust_sensors__Beta_AF7'
 'robust_sensors__Beta_AF8' 'robust_sensors__Beta_TP10'
 'robust_sensors__Gamma_TP9' 'robust_sensors__Gamma_AF7'
 'robust_sensors__Gamma_AF8' 'robust_sensors__Gamma_TP10'
 'minmax_survey__Selfreport_valence' 'minmax_survey__Selfreport_arousal'
 'minmax_survey__Selfreport_focus' 'minmax_survey__Perceived_Tiredness'
 'pass_binary__Gender' 'pass_binary__Wearing_Glasses']
MSE: 0.2547
RMSE: 0.5046
MAE: 0.4964
R-squared: -0.0237
Adjusted R-squared

## Classification models

Here we are tranining the classification models. That is, models that will classify if the predition of the resmasknet model with high or low confidence based on the selected features.

The way we define "high or low confidence resmasknet prediction" is by using a treshold. If the resmasknet model predicts and emotion with over a fixed percentage of confidence or more, then it is a high confidence prediction; otherwise, it is a low confidence one.


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)

thresholds = [0.7, 0.8, 0.9]

for thresh in thresholds:
    print("=" * 70)
    print(f" CLASSIFICATION WITH CONFIDENCE THRESHOLD = {thresh}")
    print("=" * 70)

    y_class = (result_df_regression >= thresh).astype(int)

    # Check class distribution
    if len(np.unique(y_class)) < 2:
        print(
            f"Warning: Not enough class diversity for threshold {thresh}. Skipping...\n"
        )
        continue

    # Convert all variables to float to ensure missing values are numpy safe (from pandas.NA to np.nan)
    # and also filling any created NaN to avoid downstream scikit-learn estimator failures.
    X_clf = feature_df.astype(float).fillna(0)

    X_train, X_test, y_train, y_test = train_test_split(
        X_clf, y_class, test_size=0.2, random_state=42
    )

    # Apply preprocessor perfectly matching the comment block example
    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)

    # Extract generated feature names dynamically
    feature_names_out = preprocessor.get_feature_names_out()

    # --- Random Forest Classifier ---
    print("--- Random Forest Classifier ---")
    # BIG COMPUTE: rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1)
    rf_clf = RandomForestClassifier(random_state=42, n_estimators=1, n_jobs=-1)
    rf_selector = RFECV(estimator=rf_clf, step=0.1, cv=3, scoring="accuracy")
    rf_selector.fit(X_train_scaled, y_train)

    X_train_rf = rf_selector.transform(X_train_scaled)
    X_test_rf = rf_selector.transform(X_test_scaled)
    rf_clf.fit(X_train_rf, y_train)
    y_pred_rf = rf_clf.predict(X_test_rf)

    y_pred_proba_rf = rf_clf.predict_proba(X_test_rf)[:, 1]

    print(f"Optimal features: {rf_selector.n_features_}")
    print(f"Features kept: {feature_names_out[rf_selector.support_]}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
    print(f"Macro F1 Score: {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
    try:
        roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
        print(f"ROC-AUC: {roc_auc_rf:.4f}")
    except ValueError:
        pass
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred_rf, zero_division=0))
    print("-" * 30)

    # --- Neural Network Classifier ---
    print("--- Neural Network Classifier ---")
    # BIG COMPUTE: nn_clf = MLPClassifier(random_state=42, max_iter=1000, hidden_layer_sizes=(64, 32))
    nn_clf = MLPClassifier(random_state=42, max_iter=10, hidden_layer_sizes=(2, 2))

    try:
        # Tries to remove variables if accuracy isn't hurt by > 0.001
        nn_selector = SequentialFeatureSelector(
            nn_clf,
            n_features_to_select="auto",
            tol=0.001,
            direction="backward",
            cv=3,
            scoring="accuracy",
            n_jobs=-1,
        )
        nn_selector.fit(X_train_scaled, y_train)
    except TypeError:
        # Fallback to keep 70% of features
        nn_selector = SequentialFeatureSelector(
            nn_clf,
            n_features_to_select=max(1, int(X_train_scaled.shape[1] * 0.7)),
            direction="backward",
            cv=3,
            scoring="accuracy",
            n_jobs=-1,
        )
        nn_selector.fit(X_train_scaled, y_train)

    X_train_nn = nn_selector.transform(X_train_scaled)
    X_test_nn = nn_selector.transform(X_test_scaled)

    nn_clf.fit(X_train_nn, y_train)
    y_pred_nn = nn_clf.predict(X_test_nn)

    y_pred_proba_nn = nn_clf.predict_proba(X_test_nn)[:, 1]

    print(f"Optimal features: {nn_selector.get_support().sum()}")
    print(f"Features kept: {feature_names_out[nn_selector.get_support()]}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_nn):.4f}")
    print(f"Macro F1 Score: {f1_score(y_test, y_pred_nn, average='macro'):.4f}")
    try:
        roc_auc_nn = roc_auc_score(y_test, y_pred_proba_nn)
        print(f"ROC-AUC: {roc_auc_nn:.4f}")
    except ValueError:
        pass
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred_nn, zero_division=0))
    print("\n\n")

 CLASSIFICATION WITH CONFIDENCE THRESHOLD = 0.7
--- Random Forest Classifier ---
Optimal features: 3
Features kept: ['robust_sensors__Temperature' 'robust_sensors__EDA'
 'robust_sensors__Gamma_TP9']
Accuracy: 0.6422
Macro F1 Score: 0.6404
ROC-AUC: 0.6405

Classification Report:

              precision    recall  f1-score   support

           0       0.67      0.67      0.67      2374
           1       0.62      0.61      0.61      2062

    accuracy                           0.64      4436
   macro avg       0.64      0.64      0.64      4436
weighted avg       0.64      0.64      0.64      4436

------------------------------
--- Neural Network Classifier ---
Optimal features: 29
Features kept: ['robust_sensors__EDA' 'robust_sensors__BVP' 'robust_sensors__Delta_TP9'
 'robust_sensors__Delta_AF7' 'robust_sensors__Delta_AF8'
 'robust_sensors__Delta_TP10' 'robust_sensors__Theta_TP9'
 'robust_sensors__Theta_AF7' 'robust_sensors__Theta_AF8'
 'robust_sensors__Theta_TP10' 'robust_sensors__

## WE ARE MISSING FEATURE IMPORTANCES TO KNOW WHAT FEATURES MATTER

Check this out. https://gemini.google.com/share/29bd58ad9a0e


macro avg => macro average F1

weighted avg => Weighted average F1

support => number of samples
